### Search Engine With Tools And Agents 

In [1]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun 
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper


### Wikipedia Tool (LangChain)

```
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
````

**Explanation:**

* `WikipediaAPIWrapper(...)`
  → Configures how we query Wikipedia

  * `top_k_results=1` → return only the most relevant result
  * `doc_content_chars_max=250` → limit the text length

* `WikipediaQueryRun(...)`
  → Turns the wrapper into a **tool** usable by an agent

**Summary:**
Creates a tool that lets the LLM fetch short, relevant info from Wikipedia.

```
```



### What is a Wrapper?

A **wrapper** is a layer around something (like an API or library) that makes it easier to use.

It:
- hides complexity
- standardizes how you interact with it
- adds extra configuration or formatting

**Example (Wikipedia):**

```python
api_wrapper_wiki = WikipediaAPIWrapper(...)
````

→ Instead of calling the Wikipedia API directly,
the wrapper handles:

* sending the request
* getting the result
* formatting the output

**Summary:**
A wrapper = a helper that simplifies and controls how you use something.

```
```


In [ ]:
## Using the inbuilt tool of wikipedia
# creating a wrapper around the Wikipedia API
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki


WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/opt/anaconda3/envs/venvgenai/lib/python3.10/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250))

In [3]:
wiki.name

'wikipedia'

In [4]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv

ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250))

In [5]:
arxiv.name

'arxiv'

In [6]:
#Creating the tools
tools=[wiki, arxiv]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/opt/anaconda3/envs/venvgenai/lib/python3.10/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250))]

In [7]:
## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader 
from langchain_community.vectorstores import FAISS 
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents, OpenAIEmbeddings())
vectordb

In [9]:
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x15d97ea70>, search_kwargs={})

### retriever_tool (simple explanation)

```python
from langchain_core.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith-search",
    "Search any information about Langsmith"
)
````

### What it does:

* Converts a **retriever (vector database search)** into a **tool**
* This tool can be used by an **LLM agent**

### Inputs:

* `retriever` → your vector database search engine (FAISS, Chroma, etc.)
* `"langsmith-search"` → name of the tool
* `"Search any information about Langsmith"` → description (helps the agent decide when to use it)

### Output:

* A tool that takes a user question
* Searches relevant documents
* Returns the most relevant content

### Simple idea:

Retriever → Tool → Agent can use it to search documents

```
```


In [19]:
from langchain_core.tools.retriever import create_retriever_tool
# from langchain.tools.retriever import create_retriever_tool
# from langchain_community.agent_toolkits import create_retriever_tool

retriever_tool = create_retriever_tool(retriever, "langsmith-search", "Search any information about Langsmith")
retriever_tool

StructuredTool(name='langsmith-search', description='Search any information about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x15ef4a0e0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x15ef4a170>)

In [20]:
retriever_tool.name

'langsmith-search'

In [21]:
tools=[wiki, arxiv, retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/opt/anaconda3/envs/venvgenai/lib/python3.10/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x15ef4a0e0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x15ef4a170>)]

In [22]:
## Run all this tools with Agents and LLM Models

## Tools, LLM-->AgentExecutor
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import openai
load_dotenv()
import os

groq_api_key=os.getenv("GROQ_API_KEY")
openai.api_key=os.getenv("OPENAI_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="meta-llama/llama-4-scout-17b-16e-instruct")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x15edd7d30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x15ee339a0>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

## Prompt Template (LangChain Hub)

```python
from langchain import hub
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages
````

### What this does:

* `hub.pull(...)`
  → Downloads a **prebuilt prompt template** from LangChain Hub

* `"hwchase17/openai-functions-agent"`
  → A standard prompt designed for **tool-using agents (function calling)**

### What `prompt` contains:

* System instructions for the agent
* Rules on how to use tools
* Format for reasoning and tool calls
* Chat structure (messages format)

### `prompt.messages`

* Shows the internal structure of the prompt
* Includes:

  * system message (behavior rules)
  * placeholders for user input
  * tool instructions

### Simple idea:

```text id="m8k2ld"
This prompt tells the LLM:
"How to think + when to use tools + how to respond"
```

```
```


In [ ]:
# #### Openai function agents prompt

In [27]:
## Prompt Template
# from langchain import hub
from langchain_classic import hub
prompt=hub.pull("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

## Agents (LangChain Classic)

```python
from langchain_classic.agents import create_openai_tools_agent

agent = create_openai_tools_agent(llm, tools, prompt)
agent
````

### What this does:

* Creates an **agent** that connects:

  * `llm` → the language model (brain)
  * `tools` → external tools (search, retriever, Wikipedia, etc.)
  * `prompt` → instructions for how the agent should behave

### What the agent can do:

* Decide when to use tools
* Call tools automatically
* Combine tool results with LLM reasoning
* Return a final answer

### Simple idea:

```text id="m8k2ld"
LLM (brain) + Tools (abilities) + Prompt (rules) = Agent
```

### Output:

* Returns an **agent object**
* Not a result yet — you must run it using an executor

```
```



In [29]:
####Agents
from langchain_classic.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm, tools, prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

## Agent Executor (LangChain Classic)

```python
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

agent_executor
````

### What this does:

* Wraps the **agent** and makes it runnable
* Executes the full workflow:

  * understands user question
  * decides if tools are needed
  * calls tools if necessary
  * returns final answer

### Parameters:

* `agent`
  → the reasoning system (LLM + prompt logic)

* `tools`
  → available tools (search, retriever, Wikipedia, etc.)

* `verbose=True`
  → shows step-by-step reasoning in the console:

  * thoughts
  * actions
  * tool outputs

### Simple idea:

```text id="m8k2ld"
Agent = brain
Tools = abilities
Executor = runs the system
```
### Final output:

* A runnable system that can answer questions using tools automatically

```
```


In [30]:
### Agent Executor 
from langchain_classic.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

## Running the Agent

```python
agent_executor.invoke({"input": "Tell me about Langsmith"})
````

### What this does:

* Sends a user question to the agent system
* The agent decides how to answer:

  * use tools (e.g., retriever, Wikipedia, search) if needed
  * or answer directly using the LLM

### Input format:

```python
{"input": "Tell me about Langsmith"}
```

* `"input"` → required key expected by the agent
* contains the user’s question

### What happens internally:

1. Agent receives the question
2. Decides if a tool is needed
3. Calls tool(s) if necessary
4. Collects results
5. LLM generates final answer

### Output:

* Returns a **final response object**
* Usually includes:

  * `"output"` → final answer text
  * sometimes intermediate steps (if verbose=True)

### Simple idea:

```text id="m8k2ld"
User question → Agent → Tools (if needed) → LLM → Final answer
```

```
```



In [31]:
agent_executor.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'Langsmith'}`


LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all available pages before exploring further.LangSmith is a framework-agnostic 

{'input': 'Tell me about Langsmith',
 'output': 'LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. It provides a comprehensive set of tools for observability, evaluation, prompt engineering, agent deployment, and platform setup. The platform allows users to design and deploy AI agents visually without writing code, iterate on prompts with built-in versioning and collaboration, and test prompts. LangSmith also meets the highest standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance. Users can sign up for an account and create an API key to get started with LangSmith. The platform works with many frameworks and providers, and users can choose from various quickstarts to begin building with LangSmith.'}

In [34]:
agent_executor.invoke({"input":"What is machine learning?"})



> Entering new AgentExecutor chain...
Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to learn from data, make decisions, and improve their performance over time without being explicitly programmed. 

It involves training a model on a dataset, allowing the model to learn patterns and relationships in the data, and then using that model to make predictions or take actions on new, unseen data. 

Machine learning has a wide range of applications, including image and speech recognition, natural language processing, recommender systems, and predictive analytics. 

There are several types of machine learning, including supervised learning, unsupervised learning, and reinforcement learning. 

Supervised learning involves training a model on labeled data, where the correct output is already known. 

Unsupervised learning involves training a model on unlabeled data, and the model must find patterns or s

{'input': 'What is machine learning?',
 'output': "Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to learn from data, make decisions, and improve their performance over time without being explicitly programmed. \n\nIt involves training a model on a dataset, allowing the model to learn patterns and relationships in the data, and then using that model to make predictions or take actions on new, unseen data. \n\nMachine learning has a wide range of applications, including image and speech recognition, natural language processing, recommender systems, and predictive analytics. \n\nThere are several types of machine learning, including supervised learning, unsupervised learning, and reinforcement learning. \n\nSupervised learning involves training a model on labeled data, where the correct output is already known. \n\nUnsupervised learning involves training a model on unlabeled data, and the model mu

In [33]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on cThe paper "Attention Is All You Need" (arXiv:1706.03762) introduces a new neural network architecture called Transformer, which replaces traditional recurrent neural networks (RNNs) and convolutional neural networks (CNNs) with self-attention mechanisms. The Transformer model achieves state-of-the-art results in machine translation tasks and has been widely adopted in natural language processing (NLP) applications.

> Finished chain.


{'input': "What's the paper 1706.03762 about?",
 'output': 'The paper "Attention Is All You Need" (arXiv:1706.03762) introduces a new neural network architecture called Transformer, which replaces traditional recurrent neural networks (RNNs) and convolutional neural networks (CNNs) with self-attention mechanisms. The Transformer model achieves state-of-the-art results in machine translation tasks and has been widely adopted in natural language processing (NLP) applications.'}

In [36]:
response=agent_executor.invoke({"input":"What do you know about ai in several paragraphs"})
response



> Entering new AgentExecutor chain...
Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving. AI systems can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.

There are two primary types of AI: narrow or weak AI, and general or strong AI. Narrow AI is designed to perform a specific task, such as facial recognition, language translation, or playing chess. General AI, on the other hand, is a theoretical form of AI that has not yet been achieved, which would have the ability to understand, learn, and apply knowledge across a wide range of tasks, similar to human intelligence.

The development of AI involves a range of disciplines, including computer science, mathematics, enginee

{'input': 'What do you know about ai in several paragraphs',
 'output': 'Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving. AI systems can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.\n\nThere are two primary types of AI: narrow or weak AI, and general or strong AI. Narrow AI is designed to perform a specific task, such as facial recognition, language translation, or playing chess. General AI, on the other hand, is a theoretical form of AI that has not yet been achieved, which would have the ability to understand, learn, and apply knowledge across a wide range of tasks, similar to human intelligence.\n\nThe development of AI involves a range of disciplines, including c

In [40]:
response['output']

'Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving. AI systems can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.\n\nThere are two primary types of AI: narrow or weak AI, and general or strong AI. Narrow AI is designed to perform a specific task, such as facial recognition, language translation, or playing chess. General AI, on the other hand, is a theoretical form of AI that has not yet been achieved, which would have the ability to understand, learn, and apply knowledge across a wide range of tasks, similar to human intelligence.\n\nThe development of AI involves a range of disciplines, including computer science, mathematics, engineering, and cognitive science. AI sys

In [41]:
print(response['output'])

Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving. AI systems can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.

There are two primary types of AI: narrow or weak AI, and general or strong AI. Narrow AI is designed to perform a specific task, such as facial recognition, language translation, or playing chess. General AI, on the other hand, is a theoretical form of AI that has not yet been achieved, which would have the ability to understand, learn, and apply knowledge across a wide range of tasks, similar to human intelligence.

The development of AI involves a range of disciplines, including computer science, mathematics, engineering, and cognitive science. AI systems 